# Tokenization

**Module:** 05 — LLM Fundamentals

Why we tokenize, and how BPE, WordPiece, and SentencePiece segment text for models.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain why models operate on tokens not characters/words alone
- Contrast BPE, WordPiece, and SentencePiece
- See how tokenization affects cost and code/numbers
- Debug a surprising token split


## Why Tokenize?

**Definition.** Tokenization maps raw bytes/text into a finite vocabulary of subword units the model can embed.

**Why it matters.** Open vocabularies and efficiency: characters are too long; words are too sparse.

**How it works.** Train a vocab on corpus statistics; encode/decode with that vocab at inference.

**Intuition.** Compression optimized for language statistics.

**Common pitfalls.**
- Comparing token counts across different tokenizers unfairly
- Ignoring multilingual fragmentation

**When to use.** Every LLM call—tokens drive cost and context limits.


In [ ]:
# Demo 1 — whitespace tokenizer vs chars
text = "refunds unpaid"
print("chars", len(text), "words", len(text.split()))


In [ ]:
# Demo 2 — cost from tokens
tokens, price_per_mtok = 2500, 0.5
print("$", tokens/1e6*price_per_mtok)


In [ ]:
# Demo 3 — API usage shape
print({"prompt_tokens": 1200, "completion_tokens": 180, "total_tokens": 1380})


### Try it yourself — Why Tokenize?

1. Why might 'UUID-heavy' prompts be expensive?


## BPE (Byte Pair Encoding)

**Definition.** BPE iteratively merges the most frequent adjacent pairs to grow a subword vocab from characters/bytes.

**Why it matters.** Dominates many LLM tokenizers (with byte-level variants).

**How it works.** Start from base symbols; merge until vocab size reached; encode by applying merges.

**Intuition.** Zip compression that becomes your vocabulary.

**Common pitfalls.**
- Different BPE vocabs are incompatible
- Weird splits on rare domains

**When to use.** Most GPT-style models.


In [ ]:
# Demo 1 — toy BPE merges
from collections import Counter
def get_stats(words):
    pairs = Counter()
    for word, freq in words.items():
        syms = word.split()
        for i in range(len(syms)-1):
            pairs[syms[i], syms[i+1]] += freq
    return pairs

words = {"l o w </w>": 5, "l o w e r </w>": 2, "n e w </w>": 6}
for _ in range(3):
    pairs = get_stats(words)
    best = pairs.most_common(1)[0][0]
    print("merge", best)
    bigram = " ".join(best)
    repl = "".join(best)
    words = {" ".join((" ".join(k.split())).replace(bigram, repl).split()): v for k,v in words.items()}
print(words)


In [ ]:
# Demo 2 — encode with merge table (simplified)
merges = [("l","o"), ("lo","w")]
def encode(s, merges):
    syms = list(s) + ["</w>"]
    for a,b in merges:
        i=0; out=[]
        while i < len(syms):
            if i+1 < len(syms) and syms[i]==a and syms[i+1]==b:
                out.append(a+b); i+=2
            else:
                out.append(syms[i]); i+=1
        syms = out
    return syms
print(encode("low", merges))


In [ ]:
# Demo 3 — number fragmentation sketch
print(list("12345"))  # chars; real BPE may split numbers oddly
print("tip: count tokens for numeric-heavy prompts")


### Try it yourself — BPE (Byte Pair Encoding)

1. Run a few merges manually on 'lower', 'newest'.


## WordPiece

**Definition.** WordPiece (BERT-era) also builds subwords but typically chooses merges using a likelihood criterion; uses '##' continuation markers in many implementations.

**Why it matters.** Still common in encoder stacks and some classical NLP pipelines.

**How it works.** Train vocab with WordPiece objective; encode longest-match / Viterbi variants.

**Intuition.** Similar subwords, different training/encoding details than BPE.

**Common pitfalls.**
- Assuming WordPiece == BPE
- Forgetting ## markers when detokenizing

**When to use.** BERT/many classification encoders; know it when reading older stacks.


In [ ]:
# Demo 1 — ## continuation display
pieces = ["play", "##ing", "##ful"]
print("".join(p[2:] if p.startswith("##") else p for p in pieces))


In [ ]:
# Demo 2 — longest-match greedy sketch
vocab = {"un", "able", "##able", "unbeliev", "able"}
# educational only
print("WordPiece uses trained vocab + matching, not raw whitespace")


### Try it yourself — WordPiece

1. Detokenize a ## list into a word.


## SentencePiece

**Definition.** **SentencePiece** trains tokenization directly from raw text (often language-agnostic), supporting BPE or Unigram algorithms without pre-tokenization.

**Why it matters.** Strong for multilingual models; treats text as a raw stream (▁ for spaces).

**How it works.** Train on raw corpus; encode/decode with model file; spaces become symbols.

**Intuition.** One tokenizer package to rule many languages.

**Common pitfalls.**
- Invisible space marker confusion
- Training on unclean corpora

**When to use.** Many open multilingual LLMs (T5, LLaMA-family variants historically, etc.).

| Algorithm | Pre-tokenize? | Notes |
|-----------|--------------|-------|
| BPE | Often yes (GPT) / byte-level | Merge pairs |
| WordPiece | Yes | Likelihood merges; ## |
| SentencePiece | No (raw) | BPE or Unigram |


In [ ]:
# Demo 1 — space marker intuition
raw = "hello world"
sp_style = "▁hello▁world"
print(raw, "→", sp_style)


In [ ]:
# Demo 2 — Unigram idea (likelihood) vs BPE merges
print({"BPE": "frequent pair merges", "Unigram": "probabilistic subword lexicon"})


In [ ]:
# Demo 3 — multilingual fragmentation risk
words = ["こんにちは", "Straße", "🚀"]
for w in words:
    print(w, "bytes", len(w.encode()), "python_chars", len(w))


### Try it yourself — SentencePiece

1. Why does language-agnostic raw training help multilingual corpora?


## Glossary

- **subword**: Unit between char and word
- **vocab size**: Number of distinct tokens


### Workshop drill — Tokenization (1)

Restate each section heading as a single exam-ready sentence.


In [ ]:
# Workshop drill 1 — Tokenization
headings = ['Why Tokenize?', 'BPE (Byte Pair Encoding)', 'WordPiece', 'SentencePiece']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Tokenization (2)

Change one hyperparameter/assumption in a demo and predict the effect before running.


In [ ]:
# Workshop drill 2 — Tokenization
print('prediction: ...')
print('observation: ...')
print('delta: ...')


### Workshop drill — Tokenization (3)

List production risks (cost, latency, safety, quality) for this topic.


In [ ]:
# Workshop drill 3 — Tokenization
for r in ['cost','latency','safety','quality']:
    print(f'{r}:')


### Workshop drill — Tokenization (4)

Write a tiny unit-testable helper related to the lesson and assert two cases.


In [ ]:
# Workshop drill 4 — Tokenization
def ok(x):
    return x is not None
assert ok(1) and not ok(None)
print('ok')


## Summary & Key Takeaways

- Tokens are the currency of cost and context
- BPE/WordPiece/SentencePiece differ in training and markers
- Always measure tokens on your real prompts (IDs, code, languages)

### Practice

Tokenize the same sentence with two different library tokenizers if available; compare lengths.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
